# AgeLens — 05 Validation Completion and Governance Draft

This notebook completes the remaining protocol-defined validation work:

1. Check 1 against the populated BioAge benchmark.
2. Check 2 baseline recording.
3. Check 3: survey-design-adjusted pre/post bridging PhenoAgeAccel comparison.
4. Check 4: survey-weighted missingness patterns by age group and sex.
5. Draft Validation Report and governance dispositions.

It does not modify authoritative governance documents and does not use mortality data.

In [1]:
from __future__ import annotations
from datetime import datetime, timezone
from pathlib import Path
import glob, json, shutil, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"
RUN_R_AUTOMATICALLY = True
pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 200)
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")

numpy: 2.4.6
pandas: 2.3.3
Current working directory: <PROJECT_ROOT>\notebooks


In [2]:
def find_project_root(folder_name: str = PROJECT_FOLDER_NAME) -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate
    raise FileNotFoundError(f"Could not find a parent folder named '{folder_name}'.")

def find_rscript() -> Path | None:
    on_path = shutil.which("Rscript")
    if on_path:
        return Path(on_path)
    candidates = []
    for pattern in [r"C:\Program Files\R\R-*\bin\Rscript.exe", r"C:\Program Files\R\R-*\bin\x64\Rscript.exe"]:
        candidates.extend(Path(path) for path in glob.glob(pattern))
    return sorted(candidates)[-1] if candidates else None

PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "agelens_config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
INTERIM_ROOT = PROJECT_ROOT / CONFIG["paths"]["interim_data"]
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]
SCRIPTS_ROOT = PROJECT_ROOT / "scripts"
DOCS_METHODOLOGY_ROOT = PROJECT_ROOT / "docs" / "methodology"
DOCS_GOVERNANCE_ROOT = PROJECT_ROOT / "docs" / "governance"
EXTERNAL_ROOT = PROJECT_ROOT / "data" / "external"
for path in [TABLES_ROOT,LOGS_ROOT,SCRIPTS_ROOT,DOCS_METHODOLOGY_ROOT,DOCS_GOVERNANCE_ROOT,EXTERNAL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)
PREPROCESSED_PATH = INTERIM_ROOT / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
BENCHMARK_PATH = EXTERNAL_ROOT / "bioage_phenoage_benchmark.csv"
R_SCRIPT_PATH = SCRIPTS_ROOT / "05_protocol_validation_completion.R"
print(f"Project root: {PROJECT_ROOT}")
print(f"R script expected at: {R_SCRIPT_PATH}")

Project root: <PROJECT_ROOT>
R script expected at: <PROJECT_ROOT>\scripts\05_protocol_validation_completion.R


## 1. Build the protocol validation input

In [3]:
if not PREPROCESSED_PATH.exists():
    raise FileNotFoundError(PREPROCESSED_PATH)
if not BENCHMARK_PATH.exists():
    raise FileNotFoundError(f"Missing BioAge benchmark: {BENCHMARK_PATH}")

data = pd.read_parquet(PREPROCESSED_PATH)
benchmark = pd.read_csv(BENCHMARK_PATH)
required = {"SEQN","NHANES_CYCLE","chronological_age_years","age_topcoded","RIAGENDR","RIDRETH3","WTSAF4YR","SDMVSTRA","SDMVPSU","in_fasting_subsample","complete_case_bridge_comparison","prebridge_phenoage_erratum_years","prebridge_phenoage_supplement_years","harmonized_phenoage_erratum_years","harmonized_phenoage_supplement_years","LBXSAL","LBXSCR","LBXHSCRP","LBXLYPCT","LBXMCVSI","LBXRDW","LBXSAPSI","LBXWBCSI"}
missing = sorted(required - set(data.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")
fasting = data.loc[data["in_fasting_subsample"] & data["WTSAF4YR"].notna() & data["WTSAF4YR"].gt(0)].copy()
rename = {"chronological_age_years":"age","complete_case_bridge_comparison":"bridge_complete_case","prebridge_phenoage_erratum_years":"prebridge_erratum","prebridge_phenoage_supplement_years":"prebridge_supplement","harmonized_phenoage_erratum_years":"harmonized_erratum","harmonized_phenoage_supplement_years":"harmonized_supplement"}
cols = ["SEQN","NHANES_CYCLE","chronological_age_years","age_topcoded","RIAGENDR","RIDRETH3","WTSAF4YR","SDMVSTRA","SDMVPSU","complete_case_bridge_comparison","prebridge_phenoage_erratum_years","prebridge_phenoage_supplement_years","harmonized_phenoage_erratum_years","harmonized_phenoage_supplement_years","LBXSAL","LBXSCR","LBXHSCRP","LBXLYPCT","LBXMCVSI","LBXRDW","LBXSAPSI","LBXWBCSI"]
r_input = fasting.loc[:,cols].rename(columns=rename)
r_input["age_topcoded"] = r_input["age_topcoded"].astype("int8")
r_input["bridge_complete_case"] = r_input["bridge_complete_case"].astype("int8")
if r_input.duplicated(["NHANES_CYCLE","SEQN"]).any():
    raise RuntimeError("Duplicate cycle + SEQN rows.")
R_INPUT_PATH = TABLES_ROOT / "05_protocol_validation_input.csv"
r_input.to_csv(R_INPUT_PATH,index=False)
print(f"R validation input written: {R_INPUT_PATH}")
print(f"Rows: {len(r_input):,}")
print(f"Bridge complete cases: {int(r_input['bridge_complete_case'].sum()):,}")
display(r_input.head())

R validation input written: <PROJECT_ROOT>\results\tables\05_protocol_validation_input.csv
Rows: 5,454
Bridge complete cases: 5,223


,SEQN,NHANES_CYCLE,age,age_topcoded,RIAGENDR,RIDRETH3,WTSAF4YR,SDMVSTRA,SDMVPSU,bridge_complete_case,prebridge_erratum,prebridge_supplement,harmonized_erratum,harmonized_supplement,LBXSAL,LBXSCR,LBXHSCRP,LBXLYPCT,LBXMCVSI,LBXRDW,LBXSAPSI,LBXWBCSI
1,83733,2015_2016,53.0,0,1.0,3.0,27361.171665,125.0,1.0,1,54.139170,52.702604,55.183798,53.764437,4.5,1.05,1.4,31.3,101.8,13.4,47.0,7.3
2,83734,2015_2016,78.0,0,1.0,3.0,12735.546850,131.0,1.0,1,73.701944,72.587573,74.954364,73.860620,4.5,1.12,0.6,29.9,90.8,14.7,46.0,4.4
4,83736,2015_2016,42.0,0,2.0,4.0,19089.755435,126.0,2.0,1,26.358600,24.464494,27.866282,25.997007,4.3,0.64,0.5,47.1,87.8,12.3,46.0,4.2
5,83737,2015_2016,72.0,0,2.0,1.0,12900.422816,128.0,1.0,1,74.298358,73.193810,75.214348,74.124886,4.1,1.15,2.5,31.7,92.6,14.1,83.0,6.1
9,83741,2015_2016,22.0,0,1.0,4.0,54375.644543,128.0,2.0,1,14.157710,12.062658,15.372305,13.297257,4.4,0.73,1.3,38.2,83.2,13.1,61.0,3.5


## 2. Run R survey analyses for Checks 3 and 4

In [4]:
if not R_SCRIPT_PATH.exists():
    raise FileNotFoundError(f"R script is missing: {R_SCRIPT_PATH}")
rscript = find_rscript()
print(f"Rscript: {rscript}")
if rscript is None:
    raise FileNotFoundError("Rscript was not found.")
R_STDOUT_LOG = LOGS_ROOT / "05_r_stdout.log"
R_STDERR_LOG = LOGS_ROOT / "05_r_stderr.log"
if RUN_R_AUTOMATICALLY:
    completed = subprocess.run([str(rscript),str(R_SCRIPT_PATH),str(PROJECT_ROOT)],cwd=PROJECT_ROOT,text=True,capture_output=True)
    R_STDOUT_LOG.write_text(completed.stdout or "",encoding="utf-8",errors="replace")
    R_STDERR_LOG.write_text(completed.stderr or "",encoding="utf-8",errors="replace")
    if completed.stdout:
        print("\n".join(completed.stdout.splitlines()[-200:]))
    if completed.stderr:
        print("\n--- R stderr: last 200 lines ---")
        print("\n".join(completed.stderr.splitlines()[-200:]))
    if completed.returncode != 0:
        raise RuntimeError(f"R validation failed with exit code {completed.returncode}. Logs: {R_STDOUT_LOG}, {R_STDERR_LOG}")
    print("✅ Checks 3 and 4 completed.")

Rscript: C:\Program Files\R\R-4.5.1\bin\x64\Rscript.exe

--- R stderr: last 200 lines ---
UyarÄ± mesajlarÄ±:
package 'survey' was built under R version 4.5.3 
2026-07-22 14:17:32 UTC | Starting Validation Protocol Checks 3 and 4.
2026-07-22 14:17:32 UTC | Starting Check 3.
2026-07-22 14:17:34 UTC | Check 3 completed; successful rows = 8/8.
2026-07-22 14:17:34 UTC | Starting Check 4.
2026-07-22 14:17:36 UTC | Check 4 completed; successful association rows = 16/16.
2026-07-22 14:17:36 UTC | Validation Protocol Checks 3 and 4 finished. Recorded analysis errors = 0.
Validation Protocol Checks 3 and 4 completed successfully. All weighted-missingness point estimates were independently cross-checked against positive-weight sums.
✅ Checks 3 and 4 completed.


## 3. Complete Check 1 against BioAge

In [5]:
def weighted_corr(frame: pd.DataFrame, x: str, y: str, weight: str = "WTSAF4YR") -> float:
    valid = frame[[x,y,weight]].dropna()
    valid = valid.loc[valid[weight] > 0]
    w = valid[weight].to_numpy(float); xv = valid[x].to_numpy(float); yv = valid[y].to_numpy(float)
    xm = np.average(xv,weights=w); ym = np.average(yv,weights=w)
    cov = np.average((xv-xm)*(yv-ym),weights=w)
    vx = np.average((xv-xm)**2,weights=w); vy = np.average((yv-ym)**2,weights=w)
    return float(cov/np.sqrt(vx*vy))

benchmark["SEQN"] = pd.to_numeric(benchmark["SEQN"],errors="raise").astype("Int64")
comparison = data.loc[data["complete_case_harmonized"],["SEQN","NHANES_CYCLE","chronological_age_years","age_topcoded","WTSAF4YR","harmonized_phenoage_erratum_years","harmonized_phenoage_supplement_years"]].merge(benchmark,on=["SEQN","NHANES_CYCLE"],how="inner",validate="one_to_one")
records=[]
for sample_name,mask in {"all_harmonized_complete_case":pd.Series(True,index=comparison.index),"no_topcode":~comparison["age_topcoded"]}.items():
    sample = comparison.loc[mask]
    for cycle,cf in sample.groupby("NHANES_CYCLE",observed=True):
        bio_r = weighted_corr(cf,"chronological_age_years","bioage_phenoage")
        for variant,col in {"erratum":"harmonized_phenoage_erratum_years","supplement":"harmonized_phenoage_supplement_years"}.items():
            age_r = weighted_corr(cf,"chronological_age_years",col)
            delta = age_r-bio_r
            records.append({"sample":sample_name,"cycle":cycle,"formula_variant":variant,"n":len(cf),"agelens_weighted_age_correlation":age_r,"bioage_weighted_age_correlation":bio_r,"delta_r_agelens_minus_bioage":delta,"absolute_delta_r":abs(delta),"proposed_tolerance":0.02,"diagnostic_pass":abs(delta)<0.02})
check1 = pd.DataFrame(records)
CHECK1_PATH = TABLES_ROOT / "05_check1_bioage_age_correlation.csv"
check1.to_csv(CHECK1_PATH,index=False)
display(check1.round(10))

,sample,cycle,formula_variant,n,agelens_weighted_age_correlation,bioage_weighted_age_correlation,delta_r_agelens_minus_bioage,absolute_delta_r,proposed_tolerance,diagnostic_pass
0,all_harmonized_complete_case,2015_2016,erratum,2645,0.943354,0.943307,0.000047,0.000047,0.02,True
1,all_harmonized_complete_case,2015_2016,supplement,2645,0.943354,0.943307,0.000047,0.000047,0.02,True
2,all_harmonized_complete_case,2017_2018,erratum,2578,0.945125,0.945074,0.000051,0.000051,0.02,True
3,all_harmonized_complete_case,2017_2018,supplement,2578,0.945125,0.945074,0.000051,0.000051,0.02,True
4,no_topcode,2015_2016,erratum,2524,0.938958,0.938908,0.000050,0.000050,0.02,True
5,no_topcode,2015_2016,supplement,2524,0.938958,0.938908,0.000050,0.000050,0.02,True
6,no_topcode,2017_2018,erratum,2427,0.938986,0.938929,0.000056,0.000056,0.02,True
7,no_topcode,2017_2018,supplement,2427,0.938986,0.938929,0.000056,0.000056,0.02,True


## 4. Interpret Checks 2–4

In [6]:
CHECK2_PATH = TABLES_ROOT / "03_bioage_comparison.csv"
CHECK3_PATH = TABLES_ROOT / "05_check3_bridging_effectiveness.csv"
CHECK4_CROSS_TAB_PATH = TABLES_ROOT / "05_check4_missingness_demographic_cross_tabs.csv"
CHECK4_ASSOC_PATH = TABLES_ROOT / "05_check4_missingness_association_tests.csv"
for path in [CHECK2_PATH,CHECK3_PATH,CHECK4_CROSS_TAB_PATH,CHECK4_ASSOC_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)
check2 = pd.read_csv(CHECK2_PATH)
check3 = pd.read_csv(CHECK3_PATH)
check4_cross = pd.read_csv(CHECK4_CROSS_TAB_PATH)
check4_assoc = pd.read_csv(CHECK4_ASSOC_PATH)

required_check4_audit_columns = {
    "weighted_denominator",
    "weighted_missing_numerator",
}

for frame_name, frame, count_column in [
    ("cross-tabs", check4_cross, "unweighted_missing_n"),
    ("associations", check4_assoc, "missing_n"),
]:
    missing_audit_columns = (
        required_check4_audit_columns - set(frame.columns)
    )

    if missing_audit_columns:
        raise RuntimeError(
            f"Check 4 {frame_name} output is missing audit "
            f"columns: {sorted(missing_audit_columns)}"
        )

    impossible_positive = frame.loc[
        frame[count_column].gt(0)
        & frame["weighted_missing_percent"].le(0)
    ]

    impossible_zero = frame.loc[
        frame[count_column].eq(0)
        & frame["weighted_missing_percent"].abs().gt(1e-12)
    ]

    impossible_numerator = frame.loc[
        frame[count_column].gt(0)
        & frame["weighted_missing_numerator"].le(0)
    ]

    if (
        not impossible_positive.empty
        or not impossible_zero.empty
        or not impossible_numerator.empty
    ):
        raise RuntimeError(
            "Check 4 contains internally impossible weighted "
            f"missingness results in {frame_name}. "
            "Validation reports were not regenerated."
        )

if "analysis_status" not in check3.columns:
    raise RuntimeError(
        "Check 3 output is missing analysis_status."
    )

check3_failed = check3.loc[
    ~check3["analysis_status"].eq("OK")
].copy()

if not check3_failed.empty:
    display(
        check3_failed[
            [
                "sample",
                "stage",
                "formula_variant",
                "analysis_error",
            ]
        ]
    )
    raise RuntimeError(
        "Validation Check 3 is blocking and contains "
        f"{len(check3_failed)} failed rows. "
        "Validation reports were not regenerated."
    )

def bh(values: pd.Series) -> pd.Series:
    result = pd.Series(np.nan,index=values.index,dtype=float)
    valid = values.dropna().astype(float)
    if valid.empty:
        return result
    ordered = valid.sort_values(); m=len(ordered); ranks=np.arange(1,m+1)
    adjusted = ordered.to_numpy()*m/ranks
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result.loc[ordered.index] = np.clip(adjusted,0,1)
    return result
check4_assoc["age_group_association_q_bh"] = bh(check4_assoc["age_group_association_p"])
check4_assoc["sex_association_q_bh"] = bh(check4_assoc["sex_association_p"])
check4_assoc.to_csv(CHECK4_ASSOC_PATH,index=False)
status_rows=[]
for (sample_name,variant),g in check3.groupby(["sample","formula_variant"],observed=True):
    pre=g.loc[g["stage"].eq("prebridge")].iloc[0]; post=g.loc[g["stage"].eq("harmonized")].iloc[0]
    pre_detectable=bool(pre["p_value"]<0.05)
    reduced=bool(abs(post["cohen_d_design_descriptive"])<abs(pre["cohen_d_design_descriptive"]))
    status="PASS" if pre_detectable and reduced else "INCONCLUSIVE_PRE_EFFECT_NOT_DETECTABLE" if (not pre_detectable and reduced) else "FAIL_EFFECT_NOT_REDUCED"
    status_rows.append({"sample":sample_name,"formula_variant":variant,"pre_effect_d":pre["cohen_d_design_descriptive"],"pre_p_value":pre["p_value"],"post_effect_d":post["cohen_d_design_descriptive"],"post_p_value":post["p_value"],"absolute_effect_reduction":abs(pre["cohen_d_design_descriptive"])-abs(post["cohen_d_design_descriptive"]),"status":status})
check3_status=pd.DataFrame(status_rows)
sig_missing = check4_assoc.loc[check4_assoc["age_group_association_q_bh"].lt(0.05)|check4_assoc["sex_association_q_bh"].lt(0.05)]
display(check2.round(6)); display(check3.round(6)); display(check3_status.round(6)); display(check4_assoc.round(6))
print(f"Check 4 BH-significant demographic associations: {len(sig_missing)}")

,cycle,agelens_variant,n_identical_sample,mae,rmse,mean_agelens_minus_bioage,pearson,spearman
0,2015_2016,erratum,2645,1.638000,1.679885,1.637691,1.0,1.0
1,2015_2016,supplement,2645,0.049726,0.051009,0.049726,1.0,1.0
2,2017_2018,erratum,2578,1.601316,1.644821,1.600600,1.0,1.0
3,2017_2018,supplement,2578,0.050348,0.051643,0.050337,1.0,1.0


,sample,stage,formula_variant,n,mean_accel_2015_2016,mean_accel_2017_2018,cycle_difference_2017_minus_2015,taylor_se,ci_low_95,ci_high_95,p_value,pooled_weighted_sd,cohen_d_design_descriptive,analysis_status,analysis_error
0,all_harmonized_complete_case,prebridge,erratum,5223,-0.924749,0.918195,1.842943,0.435054,0.953159,2.732728,0.000210,7.168923,0.257074,OK,NaN
1,all_harmonized_complete_case,prebridge,supplement,5223,-0.939979,0.933317,1.873296,0.442219,0.968857,2.777735,0.000210,7.286994,0.257074,OK,NaN
2,all_harmonized_complete_case,harmonized,erratum,5223,-0.252305,0.250517,0.502822,0.428386,-0.373325,1.378969,0.250041,7.018437,0.071643,OK,NaN
3,all_harmonized_complete_case,harmonized,supplement,5223,-0.256460,0.254643,0.511103,0.435441,-0.379474,1.401680,0.250041,7.134030,0.071643,OK,NaN
4,no_topcode,prebridge,erratum,4951,-0.916650,0.916421,1.833071,0.456567,0.899287,2.766856,0.000384,7.104972,0.257998,OK,NaN
5,no_topcode,prebridge,supplement,4951,-0.931747,0.931514,1.863261,0.464087,0.914098,2.812425,0.000384,7.221990,0.257998,OK,NaN
6,no_topcode,harmonized,erratum,4951,-0.241214,0.241153,0.482367,0.449067,-0.436079,1.400812,0.291611,6.952492,0.069380,OK,NaN
7,no_topcode,harmonized,supplement,4951,-0.245186,0.245125,0.490311,0.456463,-0.443261,1.423883,0.291611,7.066998,0.069380,OK,NaN


,sample,formula_variant,pre_effect_d,pre_p_value,post_effect_d,post_p_value,absolute_effect_reduction,status
0,all_harmonized_complete_case,erratum,0.257074,0.000210,0.071643,0.250041,0.185431,PASS
1,all_harmonized_complete_case,supplement,0.257074,0.000210,0.071643,0.250041,0.185431,PASS
2,no_topcode,erratum,0.257998,0.000384,0.069380,0.291611,0.188618,PASS
3,no_topcode,supplement,0.257998,0.000384,0.069380,0.291611,0.188618,PASS


,cycle,biomarker,n,missing_n,source_row_n,source_missing_n,weighted_missing_percent,weighted_denominator,weighted_missing_numerator,survey_weighted_denominator,survey_weighted_missing_numerator,max_abs_raw_minus_survey_weight,age_group_association_p,sex_association_p,little_mcar_test_run,check_scope,analysis_status,analysis_error,age_group_association_q_bh,sex_association_q_bh
0,2015_2016,albumin,2743,22,2743,22,0.704658,1.339577e+08,9.439442e+05,1.339577e+08,9.439442e+05,0.0,0.733874,0.873885,False,Design-based demographic association; Little's...,OK,NaN,0.733874,0.946709
1,2015_2016,creatinine,2743,22,2743,22,0.704658,1.339577e+08,9.439442e+05,1.339577e+08,9.439442e+05,0.0,0.733874,0.873885,False,Design-based demographic association; Little's...,OK,NaN,0.733874,0.946709
2,2015_2016,hscrp,2743,26,2743,26,0.667346,1.339577e+08,8.939619e+05,1.339577e+08,8.939619e+05,0.0,0.209305,0.266679,False,Design-based demographic association; Little's...,OK,NaN,0.544193,0.630766
3,2015_2016,lymphocyte_percent,2743,13,2743,13,0.833875,1.339577e+08,1.117040e+06,1.339577e+08,1.117040e+06,0.0,0.513865,0.485205,False,Design-based demographic association; Little's...,OK,NaN,0.733874,0.630766
4,2015_2016,mcv,2743,13,2743,13,0.833875,1.339577e+08,1.117040e+06,1.339577e+08,1.117040e+06,0.0,0.513865,0.485205,False,Design-based demographic association; Little's...,OK,NaN,0.733874,0.630766
5,2015_2016,rdw,2743,13,2743,13,0.833875,1.339577e+08,1.117040e+06,1.339577e+08,1.117040e+06,0.0,0.513865,0.485205,False,Design-based demographic association; Little's...,OK,NaN,0.733874,0.630766
6,2015_2016,alp,2743,23,2743,23,0.735939,1.339577e+08,9.858463e+05,1.339577e+08,9.858463e+05,0.0,0.651095,0.972119,False,Design-based demographic association; Little's...,OK,NaN,0.733874,0.972119
7,2015_2016,wbc,2743,13,2743,13,0.833875,1.339577e+08,1.117040e+06,1.339577e+08,1.117040e+06,0.0,0.513865,0.485205,False,Design-based demographic association; Little's...,OK,NaN,0.733874,0.630766
8,2017_2018,albumin,2711,59,2711,59,2.071380,1.362282e+08,2.821804e+06,1.362282e+08,2.821804e+06,0.0,0.031065,0.357998,False,Design-based demographic association; Little's...,OK,NaN,0.111323,0.630766
9,2017_2018,creatinine,2711,60,2711,60,2.082025,1.362282e+08,2.836306e+06,1.362282e+08,2.836306e+06,0.0,0.034253,0.350194,False,Design-based demographic association; Little's...,OK,NaN,0.111323,0.630766


Check 4 BH-significant demographic associations: 1


## 5. Generate draft reports

In [7]:
def markdown_table(frame: pd.DataFrame, max_rows: int | None = None) -> str:
    f = frame.copy()
    if max_rows is not None:
        f = f.head(max_rows)
    cols = [str(c) for c in f.columns]
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"]*len(cols)) + " |"]
    for _,row in f.iterrows():
        vals=[]
        for value in row:
            if pd.isna(value): text=""
            elif isinstance(value,(float,np.floating)): text=f"{float(value):.6g}"
            else: text=str(value)
            vals.append(text.replace("|","\\|"))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

supp = check2.loc[check2["agelens_variant"].eq("supplement")].copy()
check1_pass = bool(check1["diagnostic_pass"].all())
check2_pass = bool(np.isfinite(supp[["mae","rmse","pearson","spearman"]].to_numpy(float)).all() and supp["pearson"].ge(0.999999).all() and supp["spearman"].ge(0.999999).all())
check3_pass = bool(not check3_status.empty and check3_status["status"].eq("PASS").all())
check4_text = "PASS — no age-group or sex association survived BH correction." if sig_missing.empty else "PASS WITH DOCUMENTED LIMITATION — one or more demographic associations survived BH correction; D-006 is retained for V1 and patterns are reported."
validation_status = pd.DataFrame([
    {"check":"Check 1 — Age correlation vs BioAge","status":"PASS" if check1_pass else "FAIL","blocking":True},
    {"check":"Check 2 — Cross-implementation agreement","status":"PASS_BASELINE_ESTABLISHED" if check2_pass else "FAIL","blocking":True},
    {"check":"Check 3 — Bridging effectiveness","status":"PASS" if check3_pass else "REVIEW_REQUIRED","blocking":True},
    {"check":"Check 4 — Missingness pattern","status":check4_text,"blocking":False},
])
VALIDATION_STATUS_PATH = TABLES_ROOT / "05_validation_check_status.csv"
validation_status.to_csv(VALIDATION_STATUS_PATH,index=False)
generated_at = datetime.now(timezone.utc).isoformat()

validation_report = f"""# AgeLens Validation Report — Draft

## Document Control

| Field | Value |
| --- | --- |
| Document Title | AgeLens Validation Report |
| Project | AgeLens |
| Document ID | AL-VAL-001 |
| Version | 0.1 |
| Status | Draft — governance review required |
| Generated At (UTC) | {generated_at} |

## 1. Scope

This report records the four checks defined by `Validation_Protocol.md`. Mortality data were not used. Final results remain unauthorized while EG-004 is open and proposed dispositions are unapproved.

## 2. Sample

The harmonized complete-case sample contains 5,614 participants: 2,864 from 2015–2016 and 2,750 from 2017–2018. `WTSAF2YR / 2`, `SDMVSTRA`, and `SDMVPSU` were used. No imputation was performed.

## 3. Check Status

{markdown_table(validation_status)}

## 4. Check 1 — Age Correlation Against BioAge

{markdown_table(check1[["sample","cycle","formula_variant","n","agelens_weighted_age_correlation","bioage_weighted_age_correlation","absolute_delta_r","diagnostic_pass"]])}

The proposed tolerance `|Δr| < 0.02` was applied diagnostically and remains unapproved.

## 5. Check 2 — Cross-Implementation Agreement

{markdown_table(check2[["cycle","agelens_variant","n_identical_sample","mae","rmse","mean_agelens_minus_bioage","pearson","spearman"]])}

The Supplement pair is the clear BioAge match. The approximately 0.05-year residual difference is attributable to rounded published coefficients versus higher-precision BioAge coefficients. Seven raw BioAge package outputs were non-finite and were separately audited; the stable algebraically equivalent source-constant evaluation was used.

## 6. Check 3 — Bridging Effectiveness

{markdown_table(check3_status)}

Full design-based output is stored in `05_check3_bridging_effectiveness.csv`. Acceptance requires a detectable pre-bridging cycle effect and a smaller absolute post-bridging effect size.

## 7. Check 4 — Missingness Pattern

{markdown_table(check4_assoc[["cycle","biomarker","n","missing_n","weighted_missing_percent","age_group_association_q_bh","sex_association_q_bh"]])}

Status: {check4_text}

Glucose was excluded because absence outside the fasting subsample is structural missingness by design. Little's MCAR test was not run; this is a design-based demographic pattern assessment.

## 8. Governance Implications

- EG-002: closure recommended after direct BioAge source inspection.
- EG-010: Supplement pair recommended as canonical; Erratum retained as sensitivity.
- EG-014: retain and flag top-coded participants; report full and no-topcode sensitivity results.
- EG-004: remains open; BioAge agreement cannot independently resolve the NHANES-III creatinine training-scale question.

## 9. Release Readiness

Final scientific release is not authorized until EG-004 is dispositioned, proposed decisions are approved, and authoritative governance documents are updated in place.
"""

m15 = float(supp.loc[supp["cycle"].eq("2015_2016"),"mae"].iloc[0])
m17 = float(supp.loc[supp["cycle"].eq("2017_2018"),"mae"].iloc[0])
governance_draft = f"""# AgeLens Governance Disposition Draft

## Status

Proposed — not approved. This file does not replace `Decision_Log.md` or `Evidence_Gap_Register.md`.

## EG-002

Proposed outcome: Close. Direct source-token inspection of BioAge's `orig=TRUE` coefficients and constants was completed and recorded.

## Proposed D-010 — Formula Constant Pair

Adopt `141.50225 / 0.090165` together as the canonical published-formula pair. Retain `141.50 / 0.09165` as a named sensitivity. Do not create a hybrid pair.

Evidence: Supplement MAE was {m15:.6f} years in 2015–2016 and {m17:.6f} years in 2017–2018, versus approximately 1.6 years for the Erratum pair; Pearson and Spearman were 1.0.

Qualification: higher-precision BioAge coefficients may be retained as a software-parity diagnostic but must not silently replace source-published rounded coefficients.

## Proposed D-011 — Age Top-Coding

Retain `RIDAGEYR == 80` participants with `age_topcoded = TRUE`; do not invent exact ages. Report full-sample and no-topcode sensitivities together. Exclude top-coded records from the no-topcode face-validity correlation, but retain them in BioAge agreement checks because both implementations receive the same age input.

## EG-004

Remain Open — Core. BioAge uses the same modern creatinine input, so software agreement does not resolve whether a compensating NHANES-III-scale adjustment is scientifically required. The next analysis must quantify explicit 0.11–0.23 mg/dL compensating scenarios while preserving the unadjusted replication as reference.

## Proposed Validation Baselines

- Check 1: `|Δr| < 0.02` versus BioAge on the identical sample.
- Check 2: cycle-specific Supplement MAE values above; Pearson and Spearman 1.0.
- Check 3: use `05_check3_bridging_effectiveness.csv`.
- Check 4: demographic patterns documented; Little's MCAR not run.

## Required Updates After Approval

1. Update `Decision_Log.md` in place and bump version.
2. Update `Evidence_Gap_Register.md` in place and bump version.
3. Update `Replication_Protocol.md` and `Validation_Protocol.md`.
4. Update configuration only after decisions are approved.
"""

VALIDATION_REPORT_PATH = DOCS_METHODOLOGY_ROOT / "Validation_Report_Draft.md"
GOVERNANCE_DRAFT_PATH = DOCS_GOVERNANCE_ROOT / "Governance_Disposition_Draft.md"
VALIDATION_REPORT_PATH.write_text(validation_report,encoding="utf-8")
GOVERNANCE_DRAFT_PATH.write_text(governance_draft,encoding="utf-8")
display(validation_status)
print(f"Validation report: {VALIDATION_REPORT_PATH}")
print(f"Governance draft: {GOVERNANCE_DRAFT_PATH}")

,check,status,blocking
0,Check 1 — Age correlation vs BioAge,PASS,True
1,Check 2 — Cross-implementation agreement,PASS_BASELINE_ESTABLISHED,True
2,Check 3 — Bridging effectiveness,PASS,True
3,Check 4 — Missingness pattern,PASS WITH DOCUMENTED LIMITATION — one or more ...,False


Validation report: <PROJECT_ROOT>\docs\methodology\Validation_Report_Draft.md
Governance draft: <PROJECT_ROOT>\docs\governance\Governance_Disposition_Draft.md


## 6. Save metadata and verify

In [8]:
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "05_validation_completion.ipynb",
    "mortality_data_used": False,
    "governance_changes_applied": False,
    "little_mcar_test_run": False,
    "proposed_dispositions": {"EG-002":"close","EG-010":"proposed_D-010","EG-014":"proposed_D-011","EG-004":"remain_open_core"},
    "final_scientific_results_allowed": False,
    "outputs": [str(path.relative_to(PROJECT_ROOT)) for path in [CHECK1_PATH,CHECK3_PATH,CHECK4_CROSS_TAB_PATH,CHECK4_ASSOC_PATH,VALIDATION_STATUS_PATH,VALIDATION_REPORT_PATH,GOVERNANCE_DRAFT_PATH]],
}
METADATA_PATH = LOGS_ROOT / "05_validation_completion_metadata.json"
METADATA_PATH.write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding="utf-8")
required_outputs=[CHECK1_PATH,CHECK3_PATH,CHECK4_CROSS_TAB_PATH,CHECK4_ASSOC_PATH,VALIDATION_STATUS_PATH,VALIDATION_REPORT_PATH,GOVERNANCE_DRAFT_PATH,METADATA_PATH]
missing=[p for p in required_outputs if not p.exists()]
if missing:
    raise RuntimeError(f"Missing outputs: {missing}")
if metadata["governance_changes_applied"]:
    raise RuntimeError("Governance must not be changed automatically.")
print("✅ Validation completion outputs verified.")
print("Authoritative governance documents were not modified.")
print("Mortality data were not used.")
print("Final scientific results remain disabled.")

✅ Validation completion outputs verified.
Authoritative governance documents were not modified.
Mortality data were not used.
Final scientific results remain disabled.
